# PACKAGES

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
import base64
from  io import BytesIO
from  PIL import Image

# CONTEXTE

In [2]:
JSON_IMAGES = "/workspace/data/raw/ImageAndXML_Data_Images.json"
JSON_GT = "/workspace/data/raw/ImageAndXML_Data_GT.json"
JSON_OCR = "/workspace/data/raw/ImageAndXML_Data_OCR.json"

EXPORT_IMAGES_STREAMLIT = "/workspace/streamlit/data/raw/ImageAndXML_Data_Images.pkl"
EXPORT_GT_STREAMLIT = "/workspace/streamlit/data/raw/ImageAndXML_Data_GT.pkl"
EXPORT_OCR_STREAMLIT = "/workspace/streamlit/data/raw/ImageAndXML_Data_OCR.pkl"

 # Chargement des images -> DataFrame df_images

In [3]:
df_images = pd.read_json(JSON_IMAGES)

# Chargement des GT -> DataFrame df_gt
df_gt = pd.read_json(JSON_GT)

# Chargement des OCR-> DataFrame df_ocr
df_ocr = pd.read_json(JSON_OCR)

## PREVIEW DATAFRAME CHARGES

In [4]:
#### Preview
display(df_images.head(3))
display(df_gt.head(3))
display(df_ocr.head(3))

,image_filename,image_type,image_base64
0,0000223278.tif,tif,SUkqAHL9AACB/4H/gf+B/4H/h/+B/4H/gf+B/4H/h/+B/4...
1,0001136521.tif,tif,SUkqAAZJAQCBAIEAgQCBANoAAN6f/wAr7QAA3qf/gQCBAI...
2,0001139626.tif,tif,SUkqAKRsAQCBAIEA9QABdEGBAMUAAQtjgQD+AAExRKkAAB...


,region_id,entity,points,image_filename,image_width,image_height
0,page1_pos1,other,"651.955,733.089 678.475,733.089 678.475,848.01...",0000223278.tif,762,1000
1,page1_pos2,other,"134.214,707.833 222.609,707.833 222.609,798.76...",0000223278.tif,762,1000
2,page1_pos3,other,"505.472,603.013 606.494,603.013 606.494,662.36...",0000223278.tif,762,1000


,region_id,line_id,word_id,text,confidence,points,image_filename,image_width,image_height
0,Page1_Block1,Page1_Block1_l1,Page1_Block1_l1_w1,r>,0.165,"247,26 275,26 275,49 247,49",0000223278.tif,762,1000
1,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w1,B,0.430,"158,77 171,77 171,90 158,90",0000223278.tif,762,1000
2,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w2,AT,0.660,"175,77 210,77 210,91 175,91",0000223278.tif,762,1000


## VERIFICATION DES FORMATS POINTS -> CONVERT TO POLYGON

In [5]:
from shapely.geometry import Polygon

def parse_polygon(s):
    points = []
    for segment in s.split():
        x_str, y_str = segment.split(",")
        x = int(x_str.split(".")[0])
        y = int(y_str.split(".")[0])
        points.append((x, y))

    return Polygon(points)

df_gt["polygon"] = df_gt["points"].apply(parse_polygon)
df_ocr["polygon"] = df_ocr["points"].apply(parse_polygon)

display(df_gt.head(3))
display(df_ocr.head(3))

,region_id,entity,points,image_filename,image_width,image_height,polygon
0,page1_pos1,other,"651.955,733.089 678.475,733.089 678.475,848.01...",0000223278.tif,762,1000,"POLYGON ((651 733, 678 733, 678 848, 651 848, ..."
1,page1_pos2,other,"134.214,707.833 222.609,707.833 222.609,798.76...",0000223278.tif,762,1000,"POLYGON ((134 707, 222 707, 222 798, 134 798, ..."
2,page1_pos3,other,"505.472,603.013 606.494,603.013 606.494,662.36...",0000223278.tif,762,1000,"POLYGON ((505 603, 606 603, 606 662, 505 662, ..."


,region_id,line_id,word_id,text,confidence,points,image_filename,image_width,image_height,polygon
0,Page1_Block1,Page1_Block1_l1,Page1_Block1_l1_w1,r>,0.165,"247,26 275,26 275,49 247,49",0000223278.tif,762,1000,"POLYGON ((247 26, 275 26, 275 49, 247 49, 247 ..."
1,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w1,B,0.430,"158,77 171,77 171,90 158,90",0000223278.tif,762,1000,"POLYGON ((158 77, 171 77, 171 90, 158 90, 158 ..."
2,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w2,AT,0.660,"175,77 210,77 210,91 175,91",0000223278.tif,762,1000,"POLYGON ((175 77, 210 77, 210 91, 175 91, 175 ..."


## LIASONS ENTRE DATAFRAME

In [6]:
from shapely.strtree import STRtree
# Cherche les liaisons
def find_inclusions_strtree(df_ocr, df_gt, image_filename_lst):
    result = []

    for val in image_filename_lst:
        df_ocr_img = df_ocr[df_ocr['image_filename'] ==val ]
        df_gt_img = df_gt[df_gt['image_filename'] ==val ]

        geoms_GT = df_gt_img["polygon"].tolist()
        tree = STRtree(geoms_GT)

        index_to_region = dict(enumerate(df_gt_img["region_id"].tolist()))

        for img_filename, word_id, polygon_ocr in zip(df_ocr_img['image_filename'],df_ocr_img['word_id'], df_ocr_img['polygon']):
            candidates = tree.query(polygon_ocr, predicate=None)

            for candidate in candidates:
                if isinstance(candidate, (int, np.integer)):
                    idx = int(candidate)
                    polygon_gt = geoms_GT[idx]
                    region_id = index_to_region[idx]
                else:
                    polygon_gt = candidate
                    idx = geoms_GT.index(candidate)
                    region_id = index_to_region[idx]

                if polygon_ocr.within(polygon_gt):
                    result.append((img_filename, word_id, region_id))

    return pd.DataFrame(result, columns=["image_filename", "word_id", "region_id_GT"])

    
# recuperations de toutes les correspondances
links = find_inclusions_strtree(df_ocr, df_gt, df_images["image_filename"].unique().tolist())

# merge pour garder les liaisons
df_ocr = df_ocr.merge(links, on=['image_filename','word_id'],how="left")

df_ocr = df_ocr.merge(df_gt[['image_filename','region_id','entity','polygon']].rename(columns={"region_id": "region_id_GT", "polygon": "polygon_GT"}),
    on=["image_filename", "region_id_GT"],
    how="left"
)


In [7]:
## TRANSFORM POLYGON

In [8]:
from enum import Enum

class PolygonStatus(Enum):
    VALIDE = ("valid", "blue","green")
    INVALIDE = ("invalid", "purple", "purple")
    MANQUANT = ("missing", "red", "orange")

    @property
    def label(self):
        return self.value[0]

    @property
    def color_gt(self):
        return self.value[1]

    @property
    def color_ocr(self):
        return self.value[2]

In [9]:
def classify_polygon(p):
    if p is None or (isinstance(p, float) and pd.isna(p)):
        return PolygonStatus.MANQUANT
    if not hasattr(p, "exterior"):
         return PolygonStatus.INVALIDE
    return PolygonStatus.VALIDE

df_gt["polygon_status"] = df_gt["polygon"].apply(classify_polygon)
df_ocr["polygon_status"] = df_ocr["polygon"].apply(classify_polygon)
df_ocr["polygon_GT_status"] = df_ocr["polygon_GT"].apply(classify_polygon)

In [10]:
## VALIDATION FORMAT

In [11]:
display(df_gt.head(3))
display(df_ocr.head(3))

,region_id,entity,points,image_filename,image_width,image_height,polygon,polygon_status
0,page1_pos1,other,"651.955,733.089 678.475,733.089 678.475,848.01...",0000223278.tif,762,1000,"POLYGON ((651 733, 678 733, 678 848, 651 848, ...",PolygonStatus.VALIDE
1,page1_pos2,other,"134.214,707.833 222.609,707.833 222.609,798.76...",0000223278.tif,762,1000,"POLYGON ((134 707, 222 707, 222 798, 134 798, ...",PolygonStatus.VALIDE
2,page1_pos3,other,"505.472,603.013 606.494,603.013 606.494,662.36...",0000223278.tif,762,1000,"POLYGON ((505 603, 606 603, 606 662, 505 662, ...",PolygonStatus.VALIDE


,region_id,line_id,word_id,text,confidence,points,image_filename,image_width,image_height,polygon,region_id_GT,entity,polygon_GT,polygon_status,polygon_GT_status
0,Page1_Block1,Page1_Block1_l1,Page1_Block1_l1_w1,r>,0.165,"247,26 275,26 275,49 247,49",0000223278.tif,762,1000,"POLYGON ((247 26, 275 26, 275 49, 247 49, 247 ...",NaN,NaN,NaN,PolygonStatus.VALIDE,PolygonStatus.MANQUANT
1,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w1,B,0.430,"158,77 171,77 171,90 158,90",0000223278.tif,762,1000,"POLYGON ((158 77, 171 77, 171 90, 158 90, 158 ...",page1_pos9,supplier,"POLYGON ((93 62, 478 62, 478 127, 93 127, 93 62))",PolygonStatus.VALIDE,PolygonStatus.VALIDE
2,Page1_Block5,Page1_Block5_l1,Page1_Block5_l1_w2,AT,0.660,"175,77 210,77 210,91 175,91",0000223278.tif,762,1000,"POLYGON ((175 77, 210 77, 210 91, 175 91, 175 ...",page1_pos9,supplier,"POLYGON ((93 62, 478 62, 478 127, 93 127, 93 62))",PolygonStatus.VALIDE,PolygonStatus.VALIDE


In [12]:

df_ocr_reduit  = df_ocr[["image_filename", "region_id_GT","polygon_GT"]].rename(columns={"region_id_GT": "region_id"}).dropna(subset=["region_id"])

df_gt = df_gt.infer_objects(copy=False)

df_gt = df_gt.merge(df_ocr_reduit.assign(has_OCR=True), on=["image_filename", "region_id"],how="left")

# vérifie si colonne biien créée
if "has_OCR" not in df_gt.columns:
    df_gt["has_OCR"] = False
    # Conversion
    df_gt["has_OCR"] = df_gt["has_OCR"].astype("boolean")
else:
    # Conversion
    df_gt["has_OCR"] = df_gt["has_OCR"].astype("boolean")
    df_gt["has_OCR"] = df_gt["has_OCR"].fillna(False)


display(df_gt)

,region_id,entity,points,image_filename,image_width,image_height,polygon,polygon_status,polygon_GT,has_OCR
0,page1_pos1,other,"651.955,733.089 678.475,733.089 678.475,848.01...",0000223278.tif,762,1000,"POLYGON ((651 733, 678 733, 678 848, 651 848, ...",PolygonStatus.VALIDE,NaN,False
1,page1_pos2,other,"134.214,707.833 222.609,707.833 222.609,798.76...",0000223278.tif,762,1000,"POLYGON ((134 707, 222 707, 222 798, 134 798, ...",PolygonStatus.VALIDE,NaN,False
2,page1_pos3,other,"505.472,603.013 606.494,603.013 606.494,662.36...",0000223278.tif,762,1000,"POLYGON ((505 603, 606 603, 606 662, 505 662, ...",PolygonStatus.VALIDE,NaN,False
3,page1_pos4,other,"568.613,690.151 648.168,690.151 648.168,763.39...",0000223278.tif,762,1000,"POLYGON ((568 690, 648 690, 648 763, 568 763, ...",PolygonStatus.VALIDE,NaN,False
4,page1_pos5,invoice_info,"401.923,197.629 585.027,197.629 585.027,234.25...",0000223278.tif,762,1000,"POLYGON ((401 197, 585 197, 585 234, 401 234, ...",PolygonStatus.VALIDE,"POLYGON ((401 197, 585 197, 585 234, 401 234, ...",True
...,...,...,...,...,...,...,...,...,...,...
63795,page1_pos10,other,"37.1987,799.32 746.822,799.32 746.822,980.418 ...",tob02910.64.tif,786,1000,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",PolygonStatus.VALIDE,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",True
63796,page1_pos10,other,"37.1987,799.32 746.822,799.32 746.822,980.418 ...",tob02910.64.tif,786,1000,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",PolygonStatus.VALIDE,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",True
63797,page1_pos10,other,"37.1987,799.32 746.822,799.32 746.822,980.418 ...",tob02910.64.tif,786,1000,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",PolygonStatus.VALIDE,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",True
63798,page1_pos10,other,"37.1987,799.32 746.822,799.32 746.822,980.418 ...",tob02910.64.tif,786,1000,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",PolygonStatus.VALIDE,"POLYGON ((37 799, 746 799, 746 980, 37 980, 37...",True


In [13]:
# DATAFRAMES -> .PLK pour dashboard (streamlit)

In [14]:
# Sauvegarde des dataset pour dashboard (streamlit)
EXPORT_IMAGES_STREAMLIT = "/workspace/streamlit/data/raw/ImageAndXML_Data_Images.pkl"
EXPORT_GT_STREAMLIT = "/workspace/streamlit/data/raw/ImageAndXML_Data_GT.pkl"
EXPORT_OCR_STREAMLIT = "/workspace/streamlit/data/raw/ImageAndXML_Data_OCR.pkl"

In [15]:
df_images.to_pickle(EXPORT_IMAGES_STREAMLIT)
df_gt.to_pickle(EXPORT_GT_STREAMLIT)
df_ocr.to_pickle(EXPORT_OCR_STREAMLIT)

# DASHBOARD

In [16]:
def add_image_trace(img_b64):
    if img_b64 is None:
        return None

    # Nettoyage JSON : enlever les échappements

    # Décodage base64 → bytes
    img_bytes = base64.b64decode(img_b64)

    # Bytes → PIL → numpy
    img = Image.open(BytesIO(img_bytes)).convert("RGB")

    w,h = img.size
    
    img_np = np.array(img)
    
    fig_img = px.imshow(img_np, color_continuous_scale="gray")

    fig_img.update_coloraxes(showscale=False)
    fig_img.update_xaxes(visible=False, constrain='domain', range=[0, w])
    fig_img.update_yaxes(visible=False, range=[h, 0], scaleanchor='x', scaleratio=1)

    # Définir la taille de la figure
    fig_img.update_layout(
        title="",
        width=w,
        height=h,
        margin=dict(l=0, r=0, t=40, b=0)
    )
    return fig_img


In [17]:
def add_gt_trace(fig,df_gt_img):

    df_gt_linked = df_gt_img[df_gt_img["has_OCR"]== True]
    df_gt_unlinked = df_gt_img[df_gt_img["has_OCR"]== False]

    c_linked = len(df_gt_linked)
    c_unlinked = len(df_gt_unlinked)

    legend_gt_linked = True
    
    for idx, row in df_gt_linked.iterrows(): 
        if row["polygon_status"] == PolygonStatus.VALIDE:
            x, y = row["polygon"].exterior.xy
            hover_text = f"Entity : {row['entity']}   Region = {row['region_id']}"
            fig.add_trace(go.Scatter(x=list(x), y=list(y), 
                                        mode="lines",
                                        text=[hover_text] * len(x),
                                        hovertemplate="%{text}<extra></extra>",
                                        line=dict(color=PolygonStatus.VALIDE.color_gt), 
                                        name= f"GT ({c_linked}) ", 
                                        legendgroup="GT-linked",
                                        showlegend=legend_gt_linked))
            legend_gt_linked = False


    legend_gt_unlinked = True
    for idx, row in df_gt_unlinked.iterrows(): 
        if row["polygon_status"] == PolygonStatus.VALIDE:
            x, y = row["polygon"].exterior.xy
            hover_text = f"Entity : {row['entity']}   Region = {row['region_id']}"
            fig.add_trace(go.Scatter(x=list(x), y=list(y), 
                                        mode="lines", 
                                        text=[hover_text] * len(x),
                                        hovertemplate="%{text}<extra></extra>",
                                        line=dict(color=PolygonStatus.MANQUANT.color_gt), 
                                        name= f"GT sans OCR ({c_unlinked})", 
                                        legendgroup="GT-unlinked",
                                        showlegend=legend_gt_unlinked))
            legend_gt_unlinked = False


In [18]:
def add_ocr_trace(fig,df_ocr_img):

    legend_OCR_linked = True
    legend_OCR_unlinked = True
    legend = True
    
    c_unlinked = df_ocr_img["region_id_GT"].isna().sum()
    c_linked = len(df_ocr_img) - c_unlinked
    
    for idx, row_ocr in df_ocr_img.iterrows():

        if row_ocr["polygon_status"] == PolygonStatus.VALIDE:
            x, y = row_ocr["polygon"].exterior.xy

            if pd.isna(row_ocr["region_id_GT"]):
                hover_text = f"Texte: {row_ocr['text']}   Word = {row_ocr['word_id']}"
                name_poly=f"OCR sans GT ({c_unlinked})"
                groupname_poly ="OCR-unlinked"
                color_poly = color=PolygonStatus.MANQUANT.color_ocr

                legend =  legend_OCR_unlinked
                legend_OCR_unlinked =  False
            else:
                hover_text = f"Texte: {row_ocr['text']}   Word = {row_ocr['word_id']}   Entity: {row_ocr['entity']}"
                name_poly= f"OCR ({c_linked})"
                groupname_poly ="OCR-linked"
                color_poly = color=PolygonStatus.VALIDE.color_ocr
                
                legend =  legend_OCR_linked
                legend_OCR_linked =  False

            fig.add_trace(go.Scatter(x=list(x), y=list(y), 
                                        mode="lines", 
                                        text=[hover_text] * len(x),
                                        hovertemplate="%{text}<extra></extra>",
                                        line=dict(color=color_poly), 
                                        name= name_poly, 
                                        legendgroup=groupname_poly,
                                        showlegend=legend))
            legend_gt_unlinked = False


In [19]:
filter_key='image_filename'

# Widgets
image_dropdown = widgets.Dropdown(
    options=list(df_images[filter_key]),
    value=df_images[filter_key].iloc[0],
    description='Image :'
)

display_image_checkbox = widgets.Checkbox(value=True, description='Afficher Image')
display_gt_checkbox = widgets.Checkbox(value=True, description='Afficher GT')
display_ocr_checkbox = widgets.Checkbox(value=True, description='Afficher OCR')

output = widgets.Output()

def update_plot(change):
    with output:
        output.clear_output(wait=True)

        # Image
        if display_image_checkbox.value:
            img = df_images[df_images[filter_key] == image_dropdown.value]["image_base64"].iloc[0]
            fig = add_image_trace(img)
        else:
            fig = go.Figure()
            fig.update_yaxes(autorange='reversed')

        # GT
        if display_gt_checkbox.value:
            add_gt_trace(fig, df_gt[df_gt[filter_key] == image_dropdown.value])

        # OCR
        if display_ocr_checkbox.value:
            add_ocr_trace(fig, df_ocr[df_ocr[filter_key] == image_dropdown.value])

        display(fig)

# Lier les événements
for w in [image_dropdown, display_image_checkbox, display_gt_checkbox, display_ocr_checkbox]:
    w.observe(update_plot, names='value')

# Affichage
display(image_dropdown, display_image_checkbox, display_gt_checkbox, display_ocr_checkbox, output)

# Premier rendu
update_plot(None)


Dropdown(description='Image :', options=('0000223278.tif', '0001136521.tif', '0001139626.tif', '0001139716.tif…

Checkbox(value=True, description='Afficher Image')

Checkbox(value=True, description='Afficher GT')

Checkbox(value=True, description='Afficher OCR')

Output()